In [1]:
# ===== Cell 1: Imports =====
import os
import time
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from pathlib import Path
import xlsxwriter
from datetime import timedelta

# ===== End of Cell 1 =====

C:\Users\omrym\anaconda3\envs\deep_learn\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ===== Cell 2: Production Data Organization & Master Dataset Compilation =====

# --- 1. Configuration ---
ATR_MULTIPLIER = 1.5
EMBEDDING_DIM = 8
START_DATE = "2020-01-01"
END_DATE = "2025-12-31"

STOCKS_LIST = [
    "AAPL", "ABBV", "ADBE", "AMD", "AMT", "AMZN", "AVGO", "BAC", "BHP", "BLK",
    "BP", "COP", "COST", "CVX", "DLR", "EQIX", "FCX", "GOOGL", "GS", "INTC",
    "JNJ", "JPM", "LIN", "LLY", "META", "MRK", "MS", "NEM", "NFLX", "NVDA",
    "O", "PEP", "PFE", "PLD", "QCOM", "RIO", "SCCO", "SCHW", "SHW", "SLB",
    "SPG", "TMO", "TSLA", "TTE", "UNH", "WELL", "WFC", "XOM"
]

BASE_DATA = Path("../data")
INPUT_DIR = BASE_DATA / "raw_data"
PROCESSED_EXCEL_DIR = BASE_DATA / "formatted_data" / "atr_1.5_processed"
RAW_EXCEL_DIR = BASE_DATA / "formatted_data" / "atr_1.5_raw"
MASTER_FILE_PATH = BASE_DATA / "processed_data" / "master_dataset_M60.pt"

for d in [PROCESSED_EXCEL_DIR, RAW_EXCEL_DIR, MASTER_FILE_PATH.parent]:
    os.makedirs(d, exist_ok=True)

Z_COLS = ['O_pct', 'H_pct', 'L_pct', 'C_pct', 'MA_Mom', 'ATR_pct']
headers = ['Num', 'Date', 'Gap(O)', 'UpperW(H)', 'LowerW(L)', 'Body(C)', 'V_Money_Norm', 'MA_Mom', 'ATR_pct', ''] + \
          [f"Emb_{i+1}" for i in range(EMBEDDING_DIM)] + ['', 'Label', 'Start Target Date', 'End Target Date']

# --- 2. Helper Functions ---

def get_inflation_factor(year):
    factors = {2017: 1.35, 2018: 1.32, 2019: 1.29, 2020: 1.27, 2021: 1.21,
               2022: 1.12, 2023: 1.08, 2024: 1.03, 2025: 1.00}
    return factors.get(year, 1.0)

def compute_rigorous_features(df):
    ma_period = 14
    df['MA_Abs'] = df['Close'].rolling(window=ma_period).mean()
    tr = pd.concat([(df['High']-df['Low']), (df['High']-df['Close'].shift()).abs(), (df['Low']-df['Close'].shift()).abs()], axis=1).max(axis=1)
    df['ATR_Abs'] = tr.rolling(window=ma_period).mean()
    df['O_pct'] = (df['Open'] / df['Close'].shift(1)) - 1
    df['H_pct'] = (df['High'] / df['Open']) - 1
    df['L_pct'] = (df['Low'] / df['Open']) - 1
    df['C_pct'] = (df['Close'] / df['Open']) - 1
    df['MA_Mom'] = (df['MA_Abs'] / df['MA_Abs'].shift(1)) - 1
    df['ATR_pct'] = df['ATR_Abs'] / df['Close']
    df['Year'] = pd.to_datetime(df['Date']).dt.year
    df['V_money'] = (df['Volume'] * df['Close']) * df['Year'].apply(get_inflation_factor)
    return df

def get_window_label_rigorous(df, t, multiplier):
    atr_t = df.iloc[t]['ATR_Abs']
    open_next = df.iloc[t+1]['Open']
    buy_lvl, sell_lvl = open_next + (atr_t * multiplier), open_next - (atr_t * multiplier)
    label, is_volatile = 1, False
    for k in range(1, 6):
        idx = t + k
        h, l, c = df.iloc[idx][['High', 'Low', 'Close']]
        hb, ls = h >= buy_lvl, l <= sell_lvl
        if hb and ls:
            is_volatile = True
            label = 0 if abs(h - c) < abs(l - c) else 2
            break
        elif hb: label = 2; break
        elif ls: label = 0; break
    return label, is_volatile

def write_holiday_rows(ws, start_date, end_date, curr_row, fmts):
    delta = end_date - start_date
    for i in range(1, delta.days):
        h_date = start_date + timedelta(days=i)
        for col in range(len(headers)): ws.write(curr_row, col, "", fmts['holiday'])
        ws.write(curr_row, 1, h_date.strftime('%Y-%m-%d'), fmts['holiday'])
        ws.merge_range(curr_row, 2, curr_row, 8, "HOLIDAY / WEEKEND", fmts['holiday_text'])
        ws.write_row(curr_row, 10, [0.0]*EMBEDDING_DIM, fmts['holiday'])
        curr_row += 1
    return curr_row

# --- 3. Step 1: Global Stats Pass ---

print("📊 Pass 1: Calculating Global Stats for 48 Stocks...")
all_dfs = []
for ticker in STOCKS_LIST:
    path = INPUT_DIR / f"{ticker}_RAW.csv"
    if path.exists():
        df = compute_rigorous_features(pd.read_csv(path)).dropna()
        mask = (pd.to_datetime(df['Date']) >= START_DATE) & (pd.to_datetime(df['Date']) <= END_DATE)
        all_dfs.append(df.loc[mask])

global_all = pd.concat(all_dfs)
g_mean, g_std = global_all[Z_COLS].mean(), global_all[Z_COLS].std()
v_min, v_max = global_all['V_money'].min(), global_all['V_money'].max()

# --- 4. Step 2: Processing, Master Save, and Excel Audits ---
master_samples = []
print(f"🚀 Pass 2: Processing {len(STOCKS_LIST)} Stocks...")

for ticker in tqdm(STOCKS_LIST, colour='red'):
    path = INPUT_DIR / f"{ticker}_RAW.csv"
    if not path.exists(): continue
    df_raw = pd.read_csv(path).dropna()
    df_raw['Date'] = pd.to_datetime(df_raw['Date'])
    df_logic = compute_rigorous_features(df_raw).dropna().reset_index(drop=True)

    df_proc = df_logic.copy()
    df_proc[Z_COLS] = (df_logic[Z_COLS] - g_mean) / g_std
    df_proc['V_norm'] = (df_logic['V_money'] - v_min) / (v_max - v_min)

    for mode in ['raw', 'processed']:
        filename = f"{mode}_{ATR_MULTIPLIER}_{ticker}.xlsx"
        dir_path = RAW_EXCEL_DIR if mode == 'raw' else PROCESSED_EXCEL_DIR
        writer = pd.ExcelWriter(dir_path / filename, engine='xlsxwriter')
        wb = writer.book
        ws = wb.add_worksheet('Verification')

        fmts = {
            'head': wb.add_format({'bold': True, 'bg_color': '#D7E4BC', 'border': 1, 'align': 'center'}),
            'data': wb.add_format({'align': 'center', 'valign': 'vcenter'}),
            'special': wb.add_format({'bold': True, 'font_size': 14, 'align': 'center', 'valign': 'vcenter', 'border': 1}),
            'special_blue': wb.add_format({'bold': True, 'font_size': 14, 'align': 'center', 'valign': 'vcenter', 'border': 1, 'bg_color': '#DEEBF7'}),
            'ref_open': wb.add_format({'bold': True, 'font_size': 13, 'bg_color': '#FFFFCC', 'align': 'center', 'border': 1}),
            'holiday': wb.add_format({'bg_color': '#E1D5E7', 'align': 'center', 'valign': 'vcenter'}),
            'holiday_text': wb.add_format({'bold': True, 'font_size': 11, 'bg_color': '#E1D5E7', 'align': 'center', 'valign': 'vcenter'}),
            'buy_audit': wb.add_format({'bold': True, 'bg_color': '#C6EFCE', 'font_color': '#006100', 'align': 'center', 'valign': 'vcenter', 'border': 1, 'text_wrap': True}),
            'sell_audit': wb.add_format({'bold': True, 'bg_color': '#FFC7CE', 'font_color': '#9C0006', 'align': 'center', 'valign': 'vcenter', 'border': 1, 'text_wrap': True}),
            'target_base': wb.add_format({'bg_color': '#F2F2F2', 'align': 'center'}),
            'buy_win': wb.add_format({'bg_color': '#C6EFCE', 'font_color': '#006100', 'align': 'center', 'valign': 'vcenter', 'bold': True}),
            'sell_win': wb.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006', 'align': 'center', 'valign': 'vcenter', 'bold': True}),
            'loser_blue': wb.add_format({'bg_color': '#DEEBF7', 'font_color': '#0070C0', 'align': 'center', 'valign': 'vcenter'})
        }

        for col, val in enumerate(headers): ws.write(0, col, val, fmts['head'])
        ws.set_column('A:A', 20); ws.set_column('B:B', 15); ws.set_column('C:I', 12); ws.set_column('T:V', 30)

        curr_row, win_count = 1, 1
        source_df = df_logic if mode == 'raw' else df_proc
        final_cols = ['O_pct', 'H_pct', 'L_pct', 'C_pct', 'V_norm', 'MA_Mom', 'ATR_pct']

        for t in range(54, len(df_logic) - 5):
            if df_logic.iloc[t]['Date'] < pd.Timestamp(START_DATE) or df_logic.iloc[t]['Date'] > pd.Timestamp(END_DATE): continue

            t_open_ref = df_logic.iloc[t+1]['Open']
            t_atr_ref = df_logic.iloc[t]['ATR_Abs']
            label, is_v = get_window_label_rigorous(df_logic, t, ATR_MULTIPLIER)

            # Master Dataset Collection (Only once per ticker)
            if mode == 'processed':
                master_samples.append({
                    'x': df_proc.iloc[t-54:t+1][final_cols].values.astype(np.float32),
                    'y': label, 'ticker': ticker, 'date': df_logic.iloc[t]['Date'].strftime('%Y-%m-%d')
                })

            # Pre-calculate Triggers for Highlights
            buy_p, sell_p = t_open_ref + (t_atr_ref * ATR_MULTIPLIER), t_open_ref - (t_atr_ref * ATR_MULTIPLIER)
            trigger_k, trigger_type = None, None
            for k in range(1, 6):
                idx = t + k
                h_val, l_val = df_logic.iloc[idx]['High'], df_logic.iloc[idx]['Low']
                hb, ls = h_val >= buy_p, l_val <= sell_p
                if hb and ls: trigger_k, trigger_type = k, 'both'; break
                elif hb: trigger_k, trigger_type = k, 'buy'; break
                elif ls: trigger_k, trigger_type = k, 'sell'; break

            start_merge = curr_row
            for i in range(55):
                idx = t - 54 + i
                if i > 0: curr_row = write_holiday_rows(ws, df_logic.iloc[idx-1]['Date'], df_logic.iloc[idx]['Date'], curr_row, fmts)
                d_p = source_df.iloc[idx]
                ws.write(curr_row, 1, d_p['Date'].strftime('%Y-%m-%d'), fmts['data'])
                feat_vals = [d_p[c] if mode == 'raw' else d_p[Z_COLS[j if j<4 else j-1]] if j!=4 else d_p['V_norm']
                             for j, c in enumerate(['Open', 'High', 'Low', 'Close', 'V_money', 'MA_Mom', 'ATR_pct'])]
                ws.write_row(curr_row, 2, feat_vals, fmts['data'])
                ws.write_row(curr_row, 10, [0.0]*EMBEDDING_DIM, fmts['data'])
                curr_row += 1

            ws.merge_range(start_merge, 0, curr_row-1, 0, f"{win_count} {'VOLATILE' if is_v else ''}", fmts['special_blue'] if is_v else fmts['special'])
            ws.merge_range(start_merge, 19, curr_row-1, 19, label, fmts['special'])
            ws.merge_range(start_merge, 20, curr_row-1, 20, df_logic.iloc[t+1]['Date'].strftime('%Y-%m-%d'), fmts['special'])
            ws.merge_range(start_merge, 21, curr_row-1, 21, df_logic.iloc[t+5]['Date'].strftime('%Y-%m-%d'), fmts['special'])

            target_rows = []
            for k in range(1, 6):
                idx = t + k
                curr_row = write_holiday_rows(ws, df_logic.iloc[idx-1]['Date'], df_logic.iloc[idx]['Date'], curr_row, fmts)
                target_rows.append(curr_row)
                d_p = source_df.iloc[idx]
                feat_vals = [d_p[c] if mode == 'raw' else d_p[Z_COLS[j if j<4 else j-1]] if j!=4 else d_p['V_norm']
                             for j, c in enumerate(['Open', 'High', 'Low', 'Close', 'V_money', 'MA_Mom', 'ATR_pct'])]
                ws.write(curr_row, 1, d_p['Date'].strftime('%Y-%m-%d'), fmts['target_base'])
                for j, val in enumerate(feat_vals):
                    cell_fmt = fmts['target_base']
                    if k == 1 and j == 0: cell_fmt = fmts['ref_open']
                    if k == trigger_k:
                        if (trigger_type == 'buy' and j == 1) or (trigger_type == 'both' and label == 2 and j == 1): cell_fmt = fmts['buy_win']
                        elif (trigger_type == 'sell' and j == 2) or (trigger_type == 'both' and label == 0 and j == 2): cell_fmt = fmts['sell_win']
                        elif trigger_type == 'both' and ((label == 2 and j == 2) or (label == 0 and j == 1)): cell_fmt = fmts['loser_blue']
                    ws.write(curr_row, 2+j, val, cell_fmt)
                ws.write_row(curr_row, 10, [0.0]*EMBEDDING_DIM, fmts['target_base'])
                curr_row += 1

            ws.merge_range(target_rows[0], 19, target_rows[1], 21, f"BUY: H > {buy_p:.2f}", fmts['buy_audit'])
            ws.merge_range(target_rows[3], 19, target_rows[4], 21, f"SELL: L < {sell_p:.2f}", fmts['sell_audit'])
            curr_row += 1; win_count += 1
        writer.close()

torch.save(master_samples, MASTER_FILE_PATH)
print(f"✨ SUCCESS: Processed {len(STOCKS_LIST)} stocks. Master Dataset saved with {len(master_samples)} windows.")

# ===== End of Cell 2 =====

📊 Pass 1: Calculating Global Stats for 48 Stocks...
🚀 Pass 2: Processing 48 Stocks...


  0%|          | 0/48 [00:00<?, ?it/s]